In [10]:
import pandas as pd
import sqlite3

lib_db = sqlite3.connect('download.db')
book_catalog = pd.read_json('download.json')
kickoff_signups = pd .read_html('download.html', flavor='lxml')[0]


In [11]:
# How much is each member borrowing?
# A per-member count of checkouts — including members who haven't borrowed anything at all.
q1 = """
SELECT 
    m.member_id, 
    m.first_name, 
    m.last_name, 
    COUNT(c.checkout_id) AS total_checkouts
FROM members m
LEFT JOIN checkouts c ON m.member_id = c.member_id
GROUP BY m.member_id, m.first_name, m.last_name
ORDER BY total_checkouts DESC;
"""
print(pd.read_sql_query(q1, lib_db))

    member_id first_name last_name  total_checkouts
0        1034        Aya     Wahba               25
1        1044     Sherif     Saleh               21
2        1008       Ziad     Saleh               19
3        1010       Nour     Nabil               18
4        1027    Mostafa     Fouad               18
..        ...        ...       ...              ...
75       1063      Layla     Fouad                0
76       1064      Fares     Sabry                0
77       1066       Amir     Wahba                0
78       1069     Bassel      Adel                0
79       1078     Habiba     Osman                0

[80 rows x 4 columns]


In [13]:
# Which books match a chosen author pattern?
# A search across the catalog based on a letter or pattern you choose and record.
q2 = """
SELECT 
    book_id,
    title, 
    author
FROM books
WHERE author LIKE '%Samir%';
"""
print(pd.read_sql_query(q2, lib_db))

   book_id                  title       author
0      531  Voices in the Library  Samir Zohdy
1      532      The Last Bookmark  Samir Zohdy


In [14]:
# What are the most popular books?
# The five most-borrowed titles, ranked by how many times each was checked out.
q3 = """
SELECT 
    b.book_id, 
    b.title, 
    COUNT(c.checkout_id) AS checkout_count
FROM books b
JOIN checkouts c ON b.book_id = c.book_id
GROUP BY b.book_id, b.title
ORDER BY checkout_count DESC
LIMIT 5;
"""
print(pd.read_sql_query(q3, lib_db))

   book_id                   title  checkout_count
0      501         The Silver Kite              57
1      507   Fossils and Fireflies              55
2      513  Circuits for Beginners              46
3      519        Kites Over Cairo              38
4      525    Storms and Sailboats              25


In [15]:
# Who are the most active readers?
# The ten members who've borrowed the most books, ranked from highest to lowest.
q4 = """
SELECT 
    m.member_id, 
    m.first_name, 
    m.last_name, 
    COUNT(c.checkout_id) AS total_checkouts
FROM members m
JOIN checkouts c ON m.member_id = c.member_id
GROUP BY m.member_id, m.first_name, m.last_name
ORDER BY total_checkouts DESC
LIMIT 10;
"""
print(pd.read_sql_query(q4, lib_db))

   member_id first_name last_name  total_checkouts
0       1034        Aya     Wahba               25
1       1044     Sherif     Saleh               21
2       1008       Ziad     Saleh               19
3       1010       Nour     Nabil               18
4       1027    Mostafa     Fouad               18
5       1018      Ahmed    Shafik               17
6       1024    Youssef    Hegazy               17
7       1065       Adam     Fahmy               17
8       1030       Reem     Osman               16
9       1047       Sara    Rashad               16


In [16]:
# What does a neighborhood's activity look like further back in time?
# Checkouts from one neighborhood you choose, ordered newest to oldest, looking past the ten most recent.
q5 = """
SELECT 
    c.checkout_id, 
    c.member_id, 
    c.book_id, 
    c.checkout_date, 
    m.neighborhood
FROM checkouts c
JOIN members m ON c.member_id = m.member_id
WHERE m.neighborhood = 'Zamalek'
ORDER BY c.checkout_date DESC
LIMIT -1 OFFSET 10;
"""
print(pd.read_sql_query(q5, lib_db))

    checkout_id  member_id  book_id checkout_date neighborhood
0          9318       1067      513    2025-07-23      Zamalek
1          9320       1073      501    2025-07-19      Zamalek
2          9311       1065      507    2025-06-26      Zamalek
3          9298       1065      507    2025-06-18      Zamalek
4          9307       1072      501    2025-06-10      Zamalek
5          9333       1072      520    2025-06-03      Zamalek
6          9342       1068      501    2025-05-28      Zamalek
7          9297       1065      501    2025-04-14      Zamalek
8          9330       1072      513    2025-04-11      Zamalek
9          9288       1065      519    2025-03-20      Zamalek
10         9327       1065      513    2025-03-09      Zamalek
11         9329       1062      513    2025-03-06      Zamalek
12         9285       1071      513    2025-02-24      Zamalek
13         9341       1065      516    2025-02-19      Zamalek
14         9331       1065      525    2025-02-17      